# MT-ViT on Kaggle: prepare the data, fine-tune the model, measure before and after
**Before you press Run All:** Settings → Accelerator **GPU T4 x2** (or P100) · Internet **On** · inputs A, B and C added (see README).
Takes about 1 hour. When it finishes, download **model.zip** and **results.zip** from the Output panel.

In [ ]:
# 1. Get our code and the one library Kaggle is missing
!git clone -q https://github.com/vidush-islockedin/DeepFake-detection /kaggle/working/code
!pip install -q facenet-pytorch==2.6.0 --no-deps
%cd /kaggle/working/code
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU in Settings")

In [ ]:
# 2. Find the three datasets wherever Kaggle mounted them
from glob import glob
from pathlib import Path
def find(pattern):
    hits = sorted(glob(f"/kaggle/input/**/{pattern}", recursive=True), key=len)
    assert hits, f"Not found: {pattern}. Add that dataset with '+ Add Input'."
    return hits[0]
A = Path(find("Dataset/Train")).parent              # manjilkarki/deepfake-and-real-images
B = Path(find("real-vs-fake/train")).parent         # xhlulu/140k-real-and-fake-faces
C = find("train_sample_videos/metadata.json")       # DFDC sample videos (unseen test only)
print(A, B, C, sep="\n")

In [ ]:
# 3. MTCNN face crops -> /kaggle/temp/faces/<train|val|test>/<A|B|C>/<real|fake>/
FACES = "/kaggle/temp/faces"
!python prepare.py --src "{A}" --name A --out {FACES} --per-class 5000
!python prepare.py --src "{B}" --name B --out {FACES} --per-class 5000
!python prepare.py --dfdc "{C}" --name C --out {FACES}
!for d in {FACES}/*/*/*; do echo "$d $(ls $d | wc -l)"; done

In [ ]:
# 4. BEFORE: the downloaded pre-trained model, untouched
!python evaluate.py --model dima806/deepfake_vs_real_image_detection --data {FACES}/test --out /kaggle/working/results/before

In [ ]:
# 5. Fine-tune on A + B (C is never trained on)
!python train.py --data {FACES} --out /kaggle/working/model --epochs 2 --bs 32

In [ ]:
# 6. AFTER: our fine-tuned model on the same test sets
!python evaluate.py --model /kaggle/working/model --data {FACES}/test --out /kaggle/working/results/after
import json
for name in ("before", "after"):
    r = json.load(open(f"/kaggle/working/results/{name}/results.json"))
    print(name.upper(), {k: round(v["accuracy"], 4) for k, v in r.items() if k != "model"})

In [ ]:
# 7. Pack the two files to download
%cd /kaggle/working
!cp model/history.csv results/ && zip -qr model.zip model && zip -qr results.zip results && ls -lh *.zip